In [ ]:
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
from diffusers import DiffusionPipeline
import torch

# Load the base pipeline
base = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
).to("cuda")

# Load the refiner pipeline
refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
).to("cuda")

# Connect the base and refiner pipelines
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
).to("cuda")
pipe.unet = base.unet
pipe.vae = base.vae
pipe.text_encoder = base.text_encoder
pipe.tokenizer = base.tokenizer
pipe.scheduler = base.scheduler

# Define the prompt
prompt = "A black poor boy breaking his way into tech"

# Generate the image with the base pipeline
image = pipe(prompt=prompt, num_inference_steps=25, width=100, height=100, batch_size=1).images[0]

# Optionally, refine the image for higher quality
image = refiner(prompt=prompt, image=image.to(torch.float16), num_inference_steps=25).images[0]

# Save the generated image
image.save("new_image1.png")